# Previsão de defasagem com histórico de 3 anos

Este notebook foi refatorado para:
- fazer **full join** das abas `PEDE2022`, `PEDE2023`, `PEDE2024` por `RA`;
- construir um **target score de defasagem (0-10)** para cada aluno/ano;
- usar como base do target:
  - gap de idade (`Idade - Idade ideal da fase`, sem penalização para `FASE 8` e `FASE 9`),
  - z-score das notas (Português, Matemática, Inglês) vs média da série,
  - score de engajamento (via `IEG`);
- testar no treino não só as features principais, mas também **variáveis extras da base** (com seleção automática de colunas);
- treinar modelos para prever o **score do ano seguinte** (ex.: 2022 -> 2023, 2023 -> 2024).

In [57]:
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [58]:
# ==============================
# 1) Carga das 3 bases + full join
# ==============================
FILE_PATH = "/workspaces/Datathon-Machine-Learning-Engineering/data/BASE DE DADOS PEDE 2024 - DATATHON.xlsx"

base_2022 = pd.read_excel(FILE_PATH, sheet_name="PEDE2022")
base_2023 = pd.read_excel(FILE_PATH, sheet_name="PEDE2023")
base_2024 = pd.read_excel(FILE_PATH, sheet_name="PEDE2024")

print("Shapes brutos:")
print("2022:", base_2022.shape)
print("2023:", base_2023.shape)
print("2024:", base_2024.shape)

def add_suffix_to_columns(df, suffix, exclude_cols=("RA",)):
    rename_map = {c: c if c in exclude_cols else f"{c}_{suffix}" for c in df.columns}
    return df.rename(columns=rename_map)

df_2022 = add_suffix_to_columns(base_2022, "2022")
df_2023 = add_suffix_to_columns(base_2023, "2023")
df_2024 = add_suffix_to_columns(base_2024, "2024")

merged = (
    df_2022.merge(df_2023, on="RA", how="outer")
           .merge(df_2024, on="RA", how="outer")
)

print("\nFull join final:", merged.shape)
print("Alunos únicos (RA):", merged["RA"].nunique())

Shapes brutos:
2022: (860, 42)
2023: (1014, 48)
2024: (1156, 50)

Full join final: (1661, 138)
Alunos únicos (RA): 1661


In [59]:
# ==============================
# 2) Montar base longa (aluno-ano)
# ==============================
FASE_MAP = {
    "0": "ALFA",
    "1": "FASE 1",
    "2": "FASE 2",
    "3": "FASE 3",
    "4": "FASE 4",
    "5": "FASE 5",
    "6": "FASE 6",
    "7": "FASE 7",
    "8": "FASE 8",
    "9": "FASE 9",
}

IDADE_IDEAL_POR_FASE = {
    "ALFA": 8,
    "FASE 1": 10,
    "FASE 2": 12,
    "FASE 3": 14,
    "FASE 4": 15,
    "FASE 5": 16,
    "FASE 6": 17,
    "FASE 7": 18,
    "FASE 8": 18,
    "FASE 9": 18,
}

def norm_fase(series):
    fase_base = series.astype("string").str.strip().str.upper()
    fase_digit = fase_base.str.extract(r"(\d)", expand=False)
    return fase_digit.map(FASE_MAP).fillna(fase_base)

def pick_col(candidates, cols):
    for c in candidates:
        if c in cols:
            return c
    return None

def maybe_to_numeric(series, min_parse_ratio=0.7):
    if pd.api.types.is_numeric_dtype(series):
        return series
    cleaned = (
        series.astype("string")
        .str.replace("%", "", regex=False)
        .str.replace(",", ".", regex=False)
        .str.strip()
    )
    parsed = pd.to_numeric(cleaned, errors="coerce")
    non_null = cleaned.notna().sum()
    if non_null == 0:
        return series
    if parsed.notna().sum() / non_null >= min_parse_ratio:
        return parsed
    return series

def extract_year_frame(df_year, year, max_cat_levels=50):
    cols = df_year.columns.tolist()

    col_fase = pick_col([f"Fase_{year}"], cols)
    col_idade = pick_col([f"Idade_{year}", f"Idade 22_{year}", f"Idade_{str(year)[-2:]}"], cols)
    col_inde = pick_col([f"INDE 2024_{year}", f"INDE 2023_{year}", f"INDE 23_{year}", f"INDE 22_{year}"], cols)
    col_por = pick_col([f"Por_{year}", f"Portug_{year}", f"Português_{year}"], cols)
    col_mat = pick_col([f"Mat_{year}", f"Matem_{year}"], cols)
    col_ing = pick_col([f"Ing_{year}", f"Inglês_{year}", f"Ingles_{year}"], cols)
    col_ieg = pick_col([f"IEG_{year}"], cols)
    col_ida = pick_col([f"IDA_{year}"], cols)

    required = {
        "fase": col_fase,
        "idade": col_idade,
        "inde": col_inde,
        "portugues": col_por,
        "matematica": col_mat,
        "ingles": col_ing,
        "ieg": col_ieg,
        "ida": col_ida,
    }
    miss = [k for k, v in required.items() if v is None]
    if miss:
        raise ValueError(f"Ano {year}: colunas não encontradas -> {miss}")

    out = df_year[["RA", col_fase, col_idade, col_inde, col_por, col_mat, col_ing, col_ieg, col_ida]].copy()
    out.columns = ["RA", "Fase", "Idade", "INDE", "Portugues", "Matematica", "Ingles", "IEG", "IDA"]
    out["Ano"] = year

    out["Fase_adj"] = norm_fase(out["Fase"])
    for c in ["Idade", "INDE", "Portugues", "Matematica", "Ingles", "IEG", "IDA"]:
        out[c] = pd.to_numeric(out[c], errors="coerce")

    out["idade_ideal"] = out["Fase_adj"].map(IDADE_IDEAL_POR_FASE)
    out = out.dropna(subset=["RA", "Fase_adj", "Idade", "idade_ideal"]).copy()

    used_cols = {col_fase, col_idade, col_inde, col_por, col_mat, col_ing, col_ieg, col_ida}
    extra_cols = [c for c in cols if c.endswith(f"_{year}") and c not in used_cols]
    extra = df_year[extra_cols].copy()

    rename_extra = {}
    suffix_len = len(str(year)) + 1
    for c in extra.columns:
        base_name = c[:-suffix_len]
        rename_extra[c] = f"feat_{base_name}"
    extra = extra.rename(columns=rename_extra)
    extra = extra.loc[:, ~extra.columns.duplicated()].copy()

    for c in extra.columns:
        extra[c] = maybe_to_numeric(extra[c])

    cat_keep = []
    num_keep = []
    for c in extra.columns:
        if pd.api.types.is_numeric_dtype(extra[c]):
            num_keep.append(c)
        else:
            nunique = extra[c].nunique(dropna=True)
            if 1 < nunique <= max_cat_levels:
                cat_keep.append(c)

    extra = extra[num_keep + cat_keep]
    out = pd.concat([out, extra], axis=1)
    return out

y2022 = extract_year_frame(df_2022, 2022)
y2023 = extract_year_frame(df_2023, 2023)
y2024 = extract_year_frame(df_2024, 2024)

long_df = pd.concat([y2022, y2023, y2024], ignore_index=True)

base_cols = {"RA", "Fase", "Idade", "INDE", "Portugues", "Matematica", "Ingles", "IEG", "IDA", "Ano", "Fase_adj", "idade_ideal"}
extra_feature_cols = [c for c in long_df.columns if c not in base_cols]
print("Base longa (aluno-ano):", long_df.shape)
print("Features extras adicionadas:", len(extra_feature_cols))
long_df.head()

Base longa (aluno-ano): (3030, 54)
Features extras adicionadas: 42


,RA,Fase,Idade,INDE,Portugues,Matematica,Ingles,IEG,IDA,Ano,...,feat_INDE 22,feat_INDE 23,feat_IPP,feat_Defasagem,feat_Destaque IPV.1,feat_Pedra 2023,feat_Fase Ideal,feat_Pedra 2024,feat_Avaliador4,feat_Avaliador5
0,RA-1,7,19.0,5.783,3.5,2.7,6.0,4.1,4.0,2022.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,RA-2,7,17.0,7.055,4.5,6.3,9.7,5.2,6.8,2022.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,RA-3,7,17.0,6.591,4.0,5.8,6.9,7.9,5.6,2022.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,RA-4,7,17.0,5.951,3.5,2.8,8.7,4.5,5.0,2022.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,RA-5,7,17.0,7.427,2.9,7.0,5.7,8.6,5.2,2022.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [60]:
# ==============================
# 3) Features solicitadas + target score (0-10)
# ==============================
def zscore_group(series):
    std = series.std(ddof=0)
    if pd.isna(std) or std == 0:
        return pd.Series(np.zeros(len(series)), index=series.index)
    return (series - series.mean()) / std

work = long_df.copy()

# Gap de idade (regra de defasagem por idade)
gap_base = (work["Idade"] - work["idade_ideal"]).clip(lower=0)
work["gap_idade"] = np.where(work["Fase_adj"].isin(["FASE 8", "FASE 9"]), 0, gap_base)

# Média das notas e z-scores por ano+série (fase)
work["media_notas"] = work[["Portugues", "Matematica", "Ingles"]].mean(axis=1)

group_keys = ["Ano", "Fase_adj"]
work["z_notas_fase"] = work.groupby(group_keys)["media_notas"].transform(zscore_group)
work["z_ieg_fase"] = work.groupby(group_keys)["IEG"].transform(zscore_group)

# Componentes de risco (só penaliza quando está abaixo da média)
work["score_idade"] = work["gap_idade"]
work["score_notas"] = (-work["z_notas_fase"]).clip(lower=0) * 2.0
work["score_engajamento"] = (-work["z_ieg_fase"]).clip(lower=0) * 1.0

# Target final (0-10) sem impacto de INDE
work["target_score"] = (
    work["score_idade"] + work["score_notas"] + work["score_engajamento"]
).clip(0, 10)

print(work["target_score"].describe())
work[["RA", "Ano", "Fase_adj", "gap_idade",  "z_notas_fase", "IEG", "IDA", "score_engajamento", "target_score"]].head()

count    2619.000000
mean        1.597975
std         1.962862
min         0.000000
25%         0.000000
50%         1.000000
75%         2.404254
max        10.000000
Name: target_score, dtype: float64


,RA,Ano,Fase_adj,gap_idade,z_notas_fase,IEG,IDA,score_engajamento,target_score
0,RA-1,2022.0,FASE 7,1.0,-0.554957,4.1,4.0,1.406524,3.516437
1,RA-2,2022.0,FASE 7,0.0,0.745164,5.2,6.8,0.914241,0.914241
2,RA-3,2022.0,FASE 7,0.0,0.149928,7.9,5.6,0.000000,0.000000
3,RA-4,2022.0,FASE 7,0.0,-0.116362,4.5,5.0,1.227512,1.460235
4,RA-5,2022.0,FASE 7,0.0,-0.022377,8.6,5.2,0.000000,0.044755


In [61]:
# ==============================
# 4) Dataset temporal (ano t -> target no ano t+1)
# ==============================
feature_cols_base = [
    "gap_idade",
    "z_notas_fase",
    "IEG",
    "IDA",
    "z_ieg_fase",
]

# Seleção guiada de extras (mais interpretáveis para negócio/API)
extra_preferidas = [
    "feat_IAA",
    "feat_IPS",
    "feat_IPP",
    "feat_IPV",
    "feat_Cg",
    "feat_Cf",
    "feat_Defasagem",
]

extra_disponiveis = [c for c in work.columns if c.startswith("feat_")]
extra_feature_cols = [c for c in extra_preferidas if c in extra_disponiveis]

feature_cols_t = feature_cols_base + extra_feature_cols

def make_transition_dataset(df, ano_t, ano_t1, feature_cols):
    base_t = df.loc[df["Ano"] == ano_t, ["RA", "Fase_adj"] + feature_cols].copy()
    target_t1 = df.loc[df["Ano"] == ano_t1, ["RA", "target_score"]].copy()
    target_t1 = target_t1.rename(columns={"target_score": "target_next_year"})

    ds = base_t.merge(target_t1, on="RA", how="inner")
    ds["ano_origem"] = ano_t
    ds["transicao"] = f"{ano_t}->{ano_t1}"
    return ds

trans_22_23 = make_transition_dataset(work, 2022, 2023, feature_cols_t)
trans_23_24 = make_transition_dataset(work, 2023, 2024, feature_cols_t)

dataset_temporal = pd.concat([trans_22_23, trans_23_24], ignore_index=True)

print("Features base:", len(feature_cols_base))
print("Extras preferidas disponíveis:", len(extra_feature_cols), "->", extra_feature_cols)
print("Total de features usadas:", len(feature_cols_t))
print("Transição 2022->2023:", trans_22_23.shape)
print("Transição 2023->2024:", trans_23_24.shape)
print("Dataset temporal total:", dataset_temporal.shape)
dataset_temporal.head()

Features base: 5
Extras preferidas disponíveis: 7 -> ['feat_IAA', 'feat_IPS', 'feat_IPP', 'feat_IPV', 'feat_Cg', 'feat_Cf', 'feat_Defasagem']
Total de features usadas: 12
Transição 2022->2023: (354, 17)
Transição 2023->2024: (461, 17)
Dataset temporal total: (815, 17)


,RA,Fase_adj,gap_idade,z_notas_fase,IEG,IDA,z_ieg_fase,feat_IAA,feat_IPS,feat_IPP,feat_IPV,feat_Cg,feat_Cf,feat_Defasagem,target_next_year,ano_origem,transicao
0,RA-2,FASE 7,0.0,0.745164,5.2,6.8,-0.914241,8.8,6.3,NaN,6.778,469.0,8.0,NaN,0.00000,2022,2022->2023
1,RA-12,FASE 7,0.0,-1.761092,4.0,1.5,-1.451277,0.0,6.9,NaN,6.389,834.0,20.0,NaN,0.00000,2022,2022->2023
2,RA-13,FASE 7,0.0,-0.241675,7.4,4.7,0.070326,7.9,7.5,NaN,8.056,524.0,10.0,NaN,0.00000,2022,2022->2023
3,RA-14,FASE 7,3.0,-0.586285,8.3,4.0,0.473103,7.9,7.5,NaN,7.750,558.0,12.0,NaN,0.00000,2022,2022->2023
4,RA-19,FASE 7,1.0,1.293407,9.5,8.0,1.010140,7.9,7.5,NaN,7.000,248.0,4.0,NaN,1.34241,2022,2022->2023


In [62]:
# ==============================
# 5) Definição de X/y e split
# ==============================
model_df = dataset_temporal.dropna(subset=["target_next_year"]).copy()

X = model_df[["Fase_adj"] + feature_cols_t + ["ano_origem"]].copy()
y = model_df["target_next_year"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

# Remove colunas totalmente vazias no treino (evita warnings e simplifica)
all_null_cols = X_train.columns[X_train.isna().all()].tolist()
if all_null_cols:
    X_train = X_train.drop(columns=all_null_cols)
    X_test = X_test.drop(columns=all_null_cols)
    X = X.drop(columns=all_null_cols)
    print("Colunas removidas (100% vazias no treino):", len(all_null_cols))

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Train:", X_train.shape, "| Test:", X_test.shape)

X shape: (810, 14)
y shape: (810,)
Train: (607, 14) | Test: (203, 14)


In [63]:
# ==============================
# 6) Pipeline e modelos (versão simplificada)
# ==============================
num_cols = X.select_dtypes(include=["number", "bool"]).columns.tolist()
cat_cols = [c for c in X.columns if c not in num_cols]

transformers = []
if num_cols:
    transformers.append((
        "num",
        Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]),
        num_cols,
    ))
if cat_cols:
    transformers.append((
        "cat",
        Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
        ]),
        cat_cols,
    ))

preprocess = ColumnTransformer(transformers=transformers)

print("Numéricas:", len(num_cols), "| Categóricas:", len(cat_cols))

models = {
    "LinearRegression": LinearRegression(),
    "Ridge": Ridge(alpha=1.0, random_state=42),
    "RandomForest": RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1),
    "GradientBoosting": GradientBoostingRegressor(random_state=42),
}

print("Modelos testados:", list(models.keys()))

Numéricas: 13 | Categóricas: 1
Modelos testados: ['LinearRegression', 'Ridge', 'RandomForest', 'GradientBoosting']


In [64]:
# ==============================
# 7) Treino + seleção de features por impacto (mais assertivo)
# ==============================
from sklearn.base import clone

def build_preprocess(df):
    num_cols_local = df.select_dtypes(include=["number", "bool"]).columns.tolist()
    cat_cols_local = [c for c in df.columns if c not in num_cols_local]

    transformers_local = []
    if num_cols_local:
        transformers_local.append((
            "num",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
            ]),
            num_cols_local,
        ))
    if cat_cols_local:
        transformers_local.append((
            "cat",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
            ]),
            cat_cols_local,
        ))

    preprocess_local = ColumnTransformer(transformers=transformers_local)
    return preprocess_local, num_cols_local, cat_cols_local

def train_and_rank(Xtr, Xte, ytr, yte):
    preprocess_local, _, _ = build_preprocess(Xtr)
    rows = []
    pipes = {}

    for name, model in models.items():
        pipe = Pipeline([
            ("preprocess", preprocess_local),
            ("model", clone(model)),
        ])
        pipe.fit(Xtr, ytr)
        pred = pipe.predict(Xte)

        rows.append({
            "modelo": name,
            "MAE": round(mean_absolute_error(yte, pred), 4),
            "RMSE": round(np.sqrt(mean_squared_error(yte, pred)), 4),
            "R2": round(r2_score(yte, pred), 4),
        })
        pipes[name] = pipe

    ranked_df = pd.DataFrame(rows).sort_values(["RMSE", "MAE"], ascending=True).reset_index(drop=True)
    return ranked_df, pipes

def map_to_original_feature(transformed_name, cat_cols_local):
    if transformed_name.startswith("num__"):
        return transformed_name.replace("num__", "", 1)
    if transformed_name.startswith("cat__"):
        base = transformed_name.replace("cat__", "", 1)
        for c in cat_cols_local:
            prefix = f"{c}_"
            if base.startswith(prefix) or base == c:
                return c
        return base.split("_", 1)[0]
    return transformed_name

# 7.1 Baseline com todas as features atuais
baseline_results_df, baseline_pipes = train_and_rank(X_train, X_test, y_train, y_test)
print("Ranking baseline (todas as features):")
display(baseline_results_df)

best_model_baseline = baseline_results_df.iloc[0]["modelo"]
best_pipe_baseline = baseline_pipes[best_model_baseline]

_, _, baseline_cat_cols = build_preprocess(X_train)
feat_names = best_pipe_baseline.named_steps["preprocess"].get_feature_names_out()
best_est_baseline = best_pipe_baseline.named_steps["model"]

if hasattr(best_est_baseline, "coef_"):
    raw_impact = np.abs(best_est_baseline.coef_)
elif hasattr(best_est_baseline, "feature_importances_"):
    raw_impact = best_est_baseline.feature_importances_
else:
    raw_impact = np.zeros(len(feat_names))

impact_raw_df = pd.DataFrame({
    "feature_transformada": feat_names,
    "impacto": raw_impact,
})
impact_raw_df["variavel_original"] = impact_raw_df["feature_transformada"].apply(lambda x: map_to_original_feature(x, baseline_cat_cols))

impact_original_df = (
    impact_raw_df.groupby("variavel_original", as_index=False)["impacto"]
    .sum()
    .sort_values("impacto", ascending=False)
    .reset_index(drop=True)
)

feature_ranking = [c for c in impact_original_df["variavel_original"].tolist() if c in X_train.columns]
print("\nTop variáveis por impacto (agregado):")
display(impact_original_df.head(15))

# 7.2 Busca de melhor subconjunto de features (top-k por impacto)
min_k = 5 if len(feature_ranking) >= 5 else 1
experiments = []

for k in range(min_k, len(feature_ranking) + 1):
    selected_cols = feature_ranking[:k]
    if "Fase_adj" in X_train.columns and "Fase_adj" not in selected_cols:
        selected_cols = ["Fase_adj"] + selected_cols
    selected_cols = list(dict.fromkeys(selected_cols))

    Xtr_k = X_train[selected_cols]
    Xte_k = X_test[selected_cols]

    ranked_k, _ = train_and_rank(Xtr_k, Xte_k, y_train, y_test)
    best_k = ranked_k.iloc[0].to_dict()
    best_k["n_features"] = len(selected_cols)
    best_k["selected_columns"] = selected_cols
    experiments.append(best_k)

feature_search_df = pd.DataFrame(experiments).sort_values(["RMSE", "MAE"]).reset_index(drop=True)
print("\nMelhores combinações top-k:")
display(feature_search_df[["n_features", "modelo", "MAE", "RMSE", "R2"]].head(10))

best_row = feature_search_df.iloc[0]
final_selected_columns = best_row["selected_columns"]
final_model_name = best_row["modelo"]

# Retreino final explícito para garantir consistência entre colunas e pipeline
final_preprocess, _, final_cat_cols = build_preprocess(X_train[final_selected_columns])
final_pipeline = Pipeline([
    ("preprocess", final_preprocess),
    ("model", clone(models[final_model_name])),
])
final_pipeline.fit(X_train[final_selected_columns], y_train)

final_pred = final_pipeline.predict(X_test[final_selected_columns])
final_metrics = {
    "MAE": float(round(mean_absolute_error(y_test, final_pred), 4)),
    "RMSE": float(round(np.sqrt(mean_squared_error(y_test, final_pred)), 4)),
    "R2": float(round(r2_score(y_test, final_pred), 4)),
}

# Compatibilidade com células antigas
results_df = baseline_results_df.copy()
trained_pipelines = [(name, pipe) for name, pipe in baseline_pipes.items()]

# Avaliação por transição para o modelo final
transition_eval = pd.DataFrame({
    "transicao": model_df.loc[X_test.index, "transicao"],
    "y_real": y_test.values,
    "y_pred": final_pred,
})
transition_summary = (
    transition_eval.groupby("transicao")
    .apply(lambda d: pd.Series({
        "n": len(d),
        "MAE": round(mean_absolute_error(d["y_real"], d["y_pred"]), 4),
        "RMSE": round(np.sqrt(mean_squared_error(d["y_real"], d["y_pred"])), 4),
    }))
    .reset_index()
    .sort_values("RMSE")
    .reset_index(drop=True)
)

# Impacto final do modelo escolhido
final_feat_names = final_pipeline.named_steps["preprocess"].get_feature_names_out()
final_est = final_pipeline.named_steps["model"]
if hasattr(final_est, "coef_"):
    final_impact_values = np.abs(final_est.coef_)
elif hasattr(final_est, "feature_importances_"):
    final_impact_values = final_est.feature_importances_
else:
    final_impact_values = np.zeros(len(final_feat_names))

impact_final_raw_df = pd.DataFrame({
    "feature_transformada": final_feat_names,
    "impacto": final_impact_values,
})
impact_final_raw_df["variavel_original"] = impact_final_raw_df["feature_transformada"].apply(lambda x: map_to_original_feature(x, final_cat_cols))
impact_final_df = (
    impact_final_raw_df.groupby("variavel_original", as_index=False)["impacto"]
    .sum()
    .sort_values("impacto", ascending=False)
    .reset_index(drop=True)
)

print("\nModelo final selecionado:", final_model_name)
print("Features finais:", final_selected_columns)
print("Métricas finais:", final_metrics)
print("\nDesempenho por transição (modelo final):")
display(transition_summary)
print("\nTop impacto final:")
display(impact_final_df.head(15))

Ranking baseline (todas as features):


,modelo,MAE,RMSE,R2
0,Ridge,1.0971,1.5759,0.3024
1,LinearRegression,1.0970,1.5783,0.3002
2,RandomForest,1.1004,1.6187,0.2639
3,GradientBoosting,1.1977,1.7087,0.1799



Top variáveis por impacto (agregado):


,variavel_original,impacto
0,Fase_adj,3.619731
1,gap_idade,0.468496
2,feat_IPV,0.430571
3,z_notas_fase,0.383279
4,z_ieg_fase,0.291262
5,feat_Defasagem,0.246087
6,feat_IAA,0.153377
7,ano_origem,0.125071
8,feat_Cf,0.104014
9,IEG,0.100680



Melhores combinações top-k:


,n_features,modelo,MAE,RMSE,R2
0,5,Ridge,1.1057,1.5695,0.3080
1,6,Ridge,1.0999,1.5721,0.3058
2,12,Ridge,1.0955,1.5739,0.3041
3,11,Ridge,1.0968,1.5754,0.3028
4,13,Ridge,1.0968,1.5758,0.3025
5,14,Ridge,1.0971,1.5759,0.3024
6,8,Ridge,1.0943,1.5763,0.3020
7,9,Ridge,1.0949,1.5812,0.2977
8,10,Ridge,1.1000,1.5834,0.2957
9,7,Ridge,1.1107,1.5894,0.2903



Modelo final selecionado: Ridge
Features finais: ['Fase_adj', 'gap_idade', 'feat_IPV', 'z_notas_fase', 'z_ieg_fase']
Métricas finais: {'MAE': 1.1057, 'RMSE': 1.5695, 'R2': 0.308}

Desempenho por transição (modelo final):


,transicao,n,MAE,RMSE
0,2022->2023,88.0,0.9817,1.2744
1,2023->2024,115.0,1.2005,1.7622



Top impacto final:


,variavel_original,impacto
0,Fase_adj,4.111280
1,gap_idade,0.587601
2,z_notas_fase,0.499126
3,feat_IPV,0.393939
4,z_ieg_fase,0.218418


MAE = quanto menor melhor,
RSME = quanto menor melhor.
R2 = quanto maior melhor (0-1)

In [65]:
# 8) Resumo objetivo final (modelo otimizado por impacto)
pd.set_option("display.max_rows", 20)
pd.set_option("display.max_columns", 20)

print("Baseline (todas as features) - top 4:")
display(results_df.head(4))

print("\nBusca top-k por impacto - top 8 combinações:")
display(feature_search_df[["n_features", "modelo", "MAE", "RMSE", "R2"]].head(8))

print("\nModelo final recomendado:", final_model_name)
print("Métricas finais:", final_metrics)
print("Colunas finais para produção (CSV/API):")
print(final_selected_columns)

print("\nResumo por transição (modelo final):")
display(transition_summary)

Baseline (todas as features) - top 4:


,modelo,MAE,RMSE,R2
0,Ridge,1.0971,1.5759,0.3024
1,LinearRegression,1.0970,1.5783,0.3002
2,RandomForest,1.1004,1.6187,0.2639
3,GradientBoosting,1.1977,1.7087,0.1799



Busca top-k por impacto - top 8 combinações:


,n_features,modelo,MAE,RMSE,R2
0,5,Ridge,1.1057,1.5695,0.3080
1,6,Ridge,1.0999,1.5721,0.3058
2,12,Ridge,1.0955,1.5739,0.3041
3,11,Ridge,1.0968,1.5754,0.3028
4,13,Ridge,1.0968,1.5758,0.3025
5,14,Ridge,1.0971,1.5759,0.3024
6,8,Ridge,1.0943,1.5763,0.3020
7,9,Ridge,1.0949,1.5812,0.2977



Modelo final recomendado: Ridge
Métricas finais: {'MAE': 1.1057, 'RMSE': 1.5695, 'R2': 0.308}
Colunas finais para produção (CSV/API):
['Fase_adj', 'gap_idade', 'feat_IPV', 'z_notas_fase', 'z_ieg_fase']

Resumo por transição (modelo final):


,transicao,n,MAE,RMSE
0,2022->2023,88.0,0.9817,1.2744
1,2023->2024,115.0,1.2005,1.7622


In [66]:
# 9) Exportar modelo final otimizado para API (joblib + schema)
from pathlib import Path
import json
import joblib

api_artifacts_dir = Path("/workspaces/Datathon-Machine-Learning-Engineering/api/artifacts")
api_artifacts_dir.mkdir(parents=True, exist_ok=True)

model_path = api_artifacts_dir / "modelo_defasagem_pipeline.joblib"
schema_path = api_artifacts_dir / "modelo_defasagem_schema.json"
impact_path = api_artifacts_dir / "modelo_defasagem_impacto.csv"

joblib.dump(final_pipeline, model_path)

final_numeric_columns = [
    c for c in final_selected_columns
    if pd.api.types.is_numeric_dtype(X[c])
]
final_categorical_columns = [c for c in final_selected_columns if c not in final_numeric_columns]

schema = {
    "model_name": final_model_name,
    "target": "target_next_year",
    "required_columns": final_selected_columns,
    "numeric_columns": final_numeric_columns,
    "categorical_columns": final_categorical_columns,
    "metrics": final_metrics,
    "top_impact_features": impact_final_df.head(20).to_dict(orient="records"),
    "notes": "Envie no CSV as colunas em required_columns. Colunas extras serao ignoradas.",
}

with schema_path.open("w", encoding="utf-8") as f:
    json.dump(schema, f, ensure_ascii=False, indent=2)

impact_final_df.to_csv(impact_path, index=False)

print("Modelo exportado:", model_path)
print("Schema exportado:", schema_path)
print("Impacto exportado:", impact_path)
print("Modelo final:", final_model_name)
print("Colunas obrigatórias para CSV:", len(final_selected_columns))
final_selected_columns

Modelo exportado: /workspaces/Datathon-Machine-Learning-Engineering/api/artifacts/modelo_defasagem_pipeline.joblib
Schema exportado: /workspaces/Datathon-Machine-Learning-Engineering/api/artifacts/modelo_defasagem_schema.json
Impacto exportado: /workspaces/Datathon-Machine-Learning-Engineering/api/artifacts/modelo_defasagem_impacto.csv
Modelo final: Ridge
Colunas obrigatórias para CSV: 5


['Fase_adj', 'gap_idade', 'feat_IPV', 'z_notas_fase', 'z_ieg_fase']